# YOLOv8 for Pascal VOC 2007 Vehicle Detection

This notebook fine-tunes **YOLOv8** on **Pascal VOC 2007** for 4 vehicle classes:

- car
- bus
- motorbike
- bicycle

Designed to run on **Google Colab GPU**.

This version is formatted to stay as close as possible to the Faster R-CNN notebook structure.

In [ ]:
# Install dependencies
!pip install -q ultralytics pyyaml lxml

In [ ]:
import os
import time
import random
import shutil
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

print("Ultralytics imported successfully")

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Configuration
DATA_ROOT = "./data"
VOC_ROOT = os.path.join(DATA_ROOT, "VOCdevkit", "VOC2007")
YOLO_DATASET_ROOT = "./voc2007_yolo_vehicle"

BATCH_SIZE = 16
NUM_EPOCHS = 10
IMAGE_SIZE = 640
MODEL_NAME = "yolov8n.pt"

CLASS_NAMES = ["car", "bus", "motorbike", "bicycle"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
TARGET_CLASSES = set(CLASS_NAMES)

print(CLASS_TO_ID)

## Dataset

This notebook uses **Pascal VOC 2007** and keeps only the target vehicle classes.

Required split protocol:

- **train** = 80% of `trainval`
- **val** = 20% of `trainval`
- **test** = original VOC 2007 `test`

Images without target classes are skipped during YOLO label conversion.

In [ ]:
def ensure_voc2007_exists(voc_root):
    jpeg_dir = Path(voc_root) / "JPEGImages"
    ann_dir = Path(voc_root) / "Annotations"
    set_dir = Path(voc_root) / "ImageSets" / "Main"

    if jpeg_dir.exists() and ann_dir.exists() and set_dir.exists():
        print("VOC2007 already exists:", voc_root)
        return

    # Download VOC 2007 trainval + test directly if not present
    import urllib.request
    import tarfile

    os.makedirs(DATA_ROOT, exist_ok=True)

    urls = {
        "trainval": "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
        "test": "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar",
    }

    for name, url in urls.items():
        tar_path = os.path.join(DATA_ROOT, os.path.basename(url))
        if not os.path.exists(tar_path):
            print(f"Downloading {name}...")
            urllib.request.urlretrieve(url, tar_path)
        print(f"Extracting {name}...")
        with tarfile.open(tar_path) as tar:
            tar.extractall(path=DATA_ROOT)

    print("VOC2007 prepared at:", voc_root)

ensure_voc2007_exists(VOC_ROOT)

In [ ]:
def read_image_set(voc_root, split_name):
    split_path = Path(voc_root) / "ImageSets" / "Main" / f"{split_name}.txt"
    with open(split_path, "r") as f:
        image_ids = [line.strip() for line in f if line.strip()]
    return image_ids

full_trainval_ids = read_image_set(VOC_ROOT, "trainval")
test_ids = read_image_set(VOC_ROOT, "test")

rng = random.Random(SEED)
shuffled_ids = full_trainval_ids.copy()
rng.shuffle(shuffled_ids)

val_size = int(len(shuffled_ids) * 0.2)
val_ids = shuffled_ids[:val_size]
train_ids = shuffled_ids[val_size:]

print("Train:", len(train_ids))
print("Val:", len(val_ids))
print("Test:", len(test_ids))

In [ ]:
def voc_box_to_yolo(size, box):
    width, height = size
    xmin, ymin, xmax, ymax = box

    x_center = ((xmin + xmax) / 2.0) / width
    y_center = ((ymin + ymax) / 2.0) / height
    box_width = (xmax - xmin) / width
    box_height = (ymax - ymin) / height

    return x_center, y_center, box_width, box_height


def parse_voc_annotation(xml_file, target_classes):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    size = root.find("size")
    width = float(size.find("width").text)
    height = float(size.find("height").text)

    yolo_labels = []

    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip()
        if cls_name not in target_classes:
            continue

        difficult_tag = obj.find("difficult")
        difficult = int(difficult_tag.text) if difficult_tag is not None else 0

        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)

        if xmax <= xmin or ymax <= ymin:
            continue

        class_id = CLASS_TO_ID[cls_name]
        x_center, y_center, box_width, box_height = voc_box_to_yolo(
            (width, height), (xmin, ymin, xmax, ymax)
        )
        yolo_labels.append((class_id, x_center, y_center, box_width, box_height, difficult))

    return yolo_labels


def prepare_yolo_dataset(voc_root, output_root, train_ids, val_ids, test_ids):
    output_root = Path(output_root)

    for split in ["train", "val", "test"]:
        (output_root / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_root / "labels" / split).mkdir(parents=True, exist_ok=True)

    split_map = {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids,
    }

    kept_counts = {}

    for split, image_ids in split_map.items():
        kept = 0
        for image_id in image_ids:
            img_src = Path(voc_root) / "JPEGImages" / f"{image_id}.jpg"
            xml_src = Path(voc_root) / "Annotations" / f"{image_id}.xml"

            labels = parse_voc_annotation(xml_src, TARGET_CLASSES)
            if len(labels) == 0:
                continue

            img_dst = output_root / "images" / split / f"{image_id}.jpg"
            label_dst = output_root / "labels" / split / f"{image_id}.txt"

            shutil.copy2(img_src, img_dst)

            with open(label_dst, "w") as f:
                for class_id, x_center, y_center, box_width, box_height, difficult in labels:
                    f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}\n")

            kept += 1

        kept_counts[split] = kept
        print(f"{split}: kept {kept} images with target classes")

    data_yaml = {
        "path": str(output_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }

    yaml_path = output_root / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    print("Saved YAML to:", yaml_path)
    return str(yaml_path), kept_counts


data_yaml_path, kept_counts = prepare_yolo_dataset(
    VOC_ROOT,
    YOLO_DATASET_ROOT,
    train_ids,
    val_ids,
    test_ids
)

## Model

This section uses a pretrained **YOLOv8** checkpoint and fine-tunes it for the 4 vehicle classes.

The notebook keeps the same overall flow as the Faster R-CNN version:
- configure model
- train
- validate
- test
- visualize
- measure FPS

In [ ]:
model = YOLO(MODEL_NAME)
print("Loaded:", MODEL_NAME)

In [ ]:
train_args = {
    "data": data_yaml_path,
    "epochs": NUM_EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "project": "runs",
    "name": "yolov8_voc2007_vehicle",
    "exist_ok": True,
    "seed": SEED,
    "pretrained": True,
    "optimizer": "auto",
    "verbose": True,
}
train_args

In [ ]:
def read_results_csv(csv_path):
    import csv

    rows = []
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


def safe_float(row, *keys):
    for key in keys:
        if key in row and row[key] not in (None, ""):
            return float(row[key])
    return None

## Training

This section tracks:

- training loss
- validation mAP@0.5
- validation mAP@0.5:0.95

YOLOv8 saves these values automatically in `results.csv`.

In [ ]:
train_results = model.train(**train_args)

run_dir = Path(train_results.save_dir)
results_csv = run_dir / "results.csv"
best_path = run_dir / "weights" / "best.pt"

print("Run directory:", run_dir)
print("Results CSV:", results_csv)
print("Best weights:", best_path)

In [ ]:
rows = read_results_csv(results_csv)

train_losses = []
val_map50_list = []
val_map5095_list = []

for row in rows:
    train_box_loss = safe_float(row, "train/box_loss")
    train_cls_loss = safe_float(row, "train/cls_loss")
    train_dfl_loss = safe_float(row, "train/dfl_loss")

    total_loss = 0.0
    for value in [train_box_loss, train_cls_loss, train_dfl_loss]:
        if value is not None:
            total_loss += value

    map50 = safe_float(row, "metrics/mAP50(B)")
    map5095 = safe_float(row, "metrics/mAP50-95(B)")

    train_losses.append(total_loss)
    val_map50_list.append(map50 if map50 is not None else np.nan)
    val_map5095_list.append(map5095 if map5095 is not None else np.nan)

best_map50 = np.nanmax(val_map50_list)
print("Best val mAP@0.5:", float(best_map50))
print("Saved:", best_path)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, marker="o")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(val_map50_list, marker="o")
plt.title("Validation mAP@0.5")
plt.xlabel("Epoch")
plt.ylabel("mAP@0.5")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(val_map5095_list, marker="o")
plt.title("Validation mAP@0.5:0.95")
plt.xlabel("Epoch")
plt.ylabel("mAP@0.5:0.95")
plt.grid(True)
plt.show()

## Test Evaluation

Report all required metrics:

- **mAP@0.5**
- **mAP@0.5:0.95**

In [ ]:
best_model = YOLO(str(best_path))
test_metrics = best_model.val(data=data_yaml_path, split="test", imgsz=IMAGE_SIZE, batch=BATCH_SIZE, verbose=True)

test_map50 = float(test_metrics.box.map50)
test_map5095 = float(test_metrics.box.map)

print(f"Test mAP@0.5: {test_map50:.4f}")
print(f"Test mAP@0.5:0.95: {test_map5095:.4f}")

## Visualization

Visualize a few predictions for the report or slides.

In [ ]:
def visualize_predictions(model, image_dir, num_images=5, conf=0.25):
    image_paths = sorted(Path(image_dir).glob("*.jpg"))
    chosen = random.sample(image_paths, k=min(num_images, len(image_paths)))

    for image_path in chosen:
        results = model.predict(source=str(image_path), conf=conf, save=False, verbose=False)
        result = results[0]
        plotted = result.plot()

        plt.figure(figsize=(8, 6))
        plt.imshow(plotted[:, :, ::-1])  # BGR -> RGB
        plt.title(image_path.name)
        plt.axis("off")
        plt.show()

visualize_predictions(best_model, Path(YOLO_DATASET_ROOT) / "images" / "test", num_images=5, conf=0.25)

# Speed Measurement

Measure inference speed using:

- FPS
- seconds per image

In [ ]:
def measure_fps_yolo_fair(model, image_dir, max_images=50, imgsz=640):
    image_paths = sorted(Path(image_dir).glob("*.jpg"))[:max_images]

    total_inference_ms = 0.0
    total_postprocess_ms = 0.0

    # Bật stream=True để tối ưu bộ nhớ khi xử lý hàng loạt
    results = model.predict(source=image_paths, imgsz=imgsz, save=False, verbose=False, stream=True)

    count = 0
    for r in results:
        # Ultralytics tự động đo thời gian (tính bằng milliseconds) bên trong GPU
        # Bỏ qua thời gian đọc ảnh từ ổ cứng (preprocess)
        total_inference_ms += r.speed['inference']
        total_postprocess_ms += r.speed['postprocess']
        count += 1

    # Tính thời gian trung bình cho 1 ảnh (đổi từ ms sang giây)
    avg_time_ms = (total_inference_ms + total_postprocess_ms) / count
    sec_per_image = avg_time_ms / 1000.0

    # Tính FPS
    fps = 1.0 / sec_per_image

    return fps, sec_per_image

print("Loaded", best_path)

# Measure speed
fps, sec_per_image = measure_fps_yolo_fair(
    best_model,
    Path(YOLO_DATASET_ROOT) / "images" / "test",
    max_images=50,
    imgsz=IMAGE_SIZE
)

print(f"Tested on {fps:.0f} images (Pure Inference + NMS)")
print(f"FPS: {fps:.2f}")
print(f"Seconds / image: {sec_per_image:.4f}")

In [ ]:
print("Loaded", best_path)

In [ ]:
# Measure speed (Đã cập nhật tên hàm mới)
fps, sec_per_image = measure_fps_yolo_fair(
    best_model,
    Path(YOLO_DATASET_ROOT) / "images" / "test",
    max_images=50,
    imgsz=IMAGE_SIZE
)

print(f"Tested on 50 images (Pure Inference + NMS)")
print(f"FPS: {fps:.2f}")
print(f"Seconds / image: {sec_per_image:.4f}")